# Section 10: TensorFlow Data Pipeline
Builds manifest-driven tf.data loaders for train, val, and test splits. Applies Section 8 preprocessing to all splits and Section 9 augmentation to training only. Produces batched, prefetched datasets ready for model training.


In [1]:
%run 01_config.ipynb

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display
from augmentation_utils import augment_training_image

PREPROCESS_TARGET_SIZE = IMAGE_SIZE[0]  # 224

def preprocess_image(image_path, target_size=PREPROCESS_TARGET_SIZE, preprocess_fn=None):
    raw_bytes = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(raw_bytes, channels=3)
    shape = tf.shape(image)
    h = tf.cast(shape[0], tf.float32)
    w = tf.cast(shape[1], tf.float32)
    scale = tf.cast(target_size, tf.float32) / tf.minimum(h, w)
    new_h = tf.cast(tf.math.ceil(h * scale), tf.int32)
    new_w = tf.cast(tf.math.ceil(w * scale), tf.int32)
    image = tf.image.resize(image, [new_h, new_w], method="bilinear")
    image = tf.image.resize_with_crop_or_pad(image, target_size, target_size)
    image = tf.cast(image, tf.float32)
    if preprocess_fn is not None:
        image = preprocess_fn(image)
    else:
        image = image / 255.0
    return image

Creating output structure in: D:\SKIN CANCER/pipeline_output
Set basic random seeds to 42.
No GPU detected. Processing on CPU.


## Load Split Manifests & Validate Counts


In [2]:
d_splits = os.path.join(OUTPUT_ROOT, "splits")
d_pipeline = os.path.join(OUTPUT_ROOT, "audit_reports", "data_pipeline")
os.makedirs(d_pipeline, exist_ok=True)

df_train = pd.read_csv(os.path.join(d_splits, "train_manifest_cropped.csv"))
df_val = pd.read_csv(os.path.join(d_splits, "val_manifest_cropped.csv"))
df_test = pd.read_csv(os.path.join(d_splits, "test_manifest_cropped.csv"))

expected = {"train": 14171, "val": 2927, "test": 2957}
expected_class = {
    "train": {"NV": 8824, "MEL": 3104, "BCC": 2243},
    "val":   {"NV": 1805, "MEL": 663,  "BCC": 459},
    "test":  {"NV": 1887, "MEL": 607,  "BCC": 463}
}

splits_df = {"train": df_train, "val": df_val, "test": df_test}
errors = []

print("=== SECTION 10 INPUT DATASET VALIDATION ===")
for name, df in splits_df.items():
    actual = len(df)
    print(f"{name}: {actual} rows (Expected: {expected[name]})")
    if actual != expected[name]:
        errors.append(f"{name} row mismatch: expected {expected[name]}, got {actual}")
    counts = df["final_authoritative_label"].value_counts()
    for cls, exp_count in expected_class[name].items():
        act_count = counts.get(cls, 0)
        if act_count != exp_count:
            errors.append(f"{name}/{cls} mismatch: expected {exp_count}, got {act_count}")

if errors:
    raise ValueError(
        "Section 10 must run on the frozen split manifests, but mismatches were found:\n- "
        + "\n- ".join(errors)
    )

print("\nAll split manifest counts match expected frozen state.")


=== SECTION 10 INPUT DATASET VALIDATION ===
train: 14171 rows (Expected: 14171)
val: 2927 rows (Expected: 2927)
test: 2957 rows (Expected: 2957)

All split manifest counts match expected frozen state.


## Compute Class Weights


In [3]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = df_train["final_authoritative_label"].values
train_label_indices = np.array([CLASS_TO_INDEX[l] for l in train_labels])

weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train_label_indices)
CLASS_WEIGHTS = {i: float(w) for i, w in enumerate(weights)}

print("=== CLASS WEIGHTS (balanced, from training split) ===")
for idx, cls in INDEX_TO_CLASS.items():
    print(f"  {cls} (index {idx}): {CLASS_WEIGHTS[idx]:.4f}")


=== CLASS WEIGHTS (balanced, from training split) ===
  NV (index 0): 0.5353
  MEL (index 1): 1.5218
  BCC (index 2): 2.1060


## Define Pipeline Constants


In [4]:
PIPELINE_BATCH_SIZE = BATCH_SIZE  # 32 from config
PIPELINE_SHUFFLE_BUFFER = 4096
PIPELINE_PREFETCH = tf.data.AUTOTUNE
PIPELINE_NUM_PARALLEL = tf.data.AUTOTUNE

print("=== DATA PIPELINE CONSTANTS ===")
print(f"Batch size: {PIPELINE_BATCH_SIZE}")
print(f"Shuffle buffer (train only): {PIPELINE_SHUFFLE_BUFFER}")
print(f"Prefetch: AUTOTUNE")
print(f"Parallel map calls: AUTOTUNE")


=== DATA PIPELINE CONSTANTS ===
Batch size: 32
Shuffle buffer (train only): 4096
Prefetch: AUTOTUNE
Parallel map calls: AUTOTUNE


## Build tf.data Datasets


In [5]:
PIPELINE_BATCH_SIZE = BATCH_SIZE  # 32
PIPELINE_SHUFFLE_BUFFER = 4096
PIPELINE_PREFETCH = tf.data.AUTOTUNE
PIPELINE_NUM_PARALLEL = tf.data.AUTOTUNE

def build_dataset(df, is_training=False, batch_size=PIPELINE_BATCH_SIZE):
    paths = df["full_path"].values
    labels = np.array([CLASS_TO_INDEX[l] for l in df["final_authoritative_label"].values])

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if is_training:
        ds = ds.shuffle(PIPELINE_SHUFFLE_BUFFER, seed=RANDOM_SEED, reshuffle_each_iteration=True)

    def load_and_preprocess(path, label):
        image = preprocess_image(path, preprocess_fn=None)
        return image, label

    ds = ds.map(load_and_preprocess, num_parallel_calls=PIPELINE_NUM_PARALLEL)

    if is_training:
        def augment_fn(image, label):
            image = augment_training_image(image)
            return image, label
        ds = ds.map(augment_fn, num_parallel_calls=PIPELINE_NUM_PARALLEL)

    def apply_backbone_preprocess(image, label):
        image = image * 255.0
        image = tf.keras.applications.efficientnet.preprocess_input(image)
        return image, label

    ds = ds.map(apply_backbone_preprocess, num_parallel_calls=PIPELINE_NUM_PARALLEL)

    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(PIPELINE_PREFETCH)

    return ds


print("Building datasets...")
print("Order: Load -> [0,1] -> Augment (train only) -> EfficientNet preprocess")
ds_train = build_dataset(df_train, is_training=True)
ds_val = build_dataset(df_val, is_training=False)
ds_test = build_dataset(df_test, is_training=False)
print("Datasets built.")

Building datasets...
Order: Load -> [0,1] -> Augment (train only) -> EfficientNet preprocess
Datasets built.


## Validation Check 1: Batch Shape, Dtype, Label Range


In [6]:
validation_results = []

print("=== BATCH VALIDATION ===")
for name, ds in [("train", ds_train), ("val", ds_val), ("test", ds_test)]:
    for images, labels in ds.take(1):
        img_shape = images.shape
        img_dtype = images.dtype
        lbl_shape = labels.shape
        lbl_dtype = labels.dtype
        lbl_min = int(tf.reduce_min(labels).numpy())
        lbl_max = int(tf.reduce_max(labels).numpy())
        img_min = float(tf.reduce_min(images).numpy())
        img_max = float(tf.reduce_max(images).numpy())

        print(f"\n{name}:")
        print(f"  Image batch: shape={img_shape}, dtype={img_dtype}, range=[{img_min:.4f}, {img_max:.4f}]")
        print(f"  Label batch: shape={lbl_shape}, dtype={lbl_dtype}, range=[{lbl_min}, {lbl_max}]")

        shape_ok = (len(img_shape) == 4 and img_shape[1] == PREPROCESS_TARGET_SIZE
                    and img_shape[2] == PREPROCESS_TARGET_SIZE and img_shape[3] == 3)
        dtype_ok = (img_dtype == tf.float32)
        range_ok = (img_min >= 0.0 and img_max <= 1.0)
        label_ok = (lbl_min >= 0 and lbl_max <= 2)

        validation_results.append({"check": f"{name}_batch_shape", "expected": f"(B,{PREPROCESS_TARGET_SIZE},{PREPROCESS_TARGET_SIZE},3)", "actual": str(tuple(img_shape.as_list())), "pass": shape_ok})
        validation_results.append({"check": f"{name}_batch_dtype", "expected": "float32", "actual": str(img_dtype.name), "pass": dtype_ok})
        validation_results.append({"check": f"{name}_batch_range", "expected": "[0.0, 1.0]", "actual": f"[{img_min:.4f}, {img_max:.4f}]", "pass": range_ok})
        validation_results.append({"check": f"{name}_label_range", "expected": "[0, 2]", "actual": f"[{lbl_min}, {lbl_max}]", "pass": label_ok})


=== BATCH VALIDATION ===

train:
  Image batch: shape=(32, 224, 224, 3), dtype=<dtype: 'float32'>, range=[0.0000, 255.0000]
  Label batch: shape=(32,), dtype=<dtype: 'int64'>, range=[0, 0]

val:
  Image batch: shape=(32, 224, 224, 3), dtype=<dtype: 'float32'>, range=[0.0000, 255.0000]
  Label batch: shape=(32,), dtype=<dtype: 'int64'>, range=[0, 0]

test:
  Image batch: shape=(32, 224, 224, 3), dtype=<dtype: 'float32'>, range=[0.0000, 255.0000]
  Label batch: shape=(32,), dtype=<dtype: 'int64'>, range=[0, 0]


## Validation Check 2: Total Sample Count Across Batches


In [7]:
print("Counting total samples across all batches per split...")

for name, ds, exp_count in [("train", ds_train, expected["train"]),
                             ("val", ds_val, expected["val"]),
                             ("test", ds_test, expected["test"])]:
    total = 0
    n_batches = 0
    for images, labels in ds:
        total += images.shape[0]
        n_batches += 1

    count_ok = (total == exp_count)
    print(f"{name}: {total} samples in {n_batches} batches (Expected: {exp_count}). Pass: {count_ok}")

    validation_results.append({"check": f"{name}_total_sample_count", "expected": str(exp_count), "actual": str(total), "pass": count_ok})

print("\nTotal sample count validation complete.")


Counting total samples across all batches per split...
train: 14171 samples in 443 batches (Expected: 14171). Pass: True
val: 2927 samples in 92 batches (Expected: 2927). Pass: True
test: 2957 samples in 93 batches (Expected: 2957). Pass: True

Total sample count validation complete.


## Validation Check 3: Label Distribution in First Epoch


In [8]:
print("Checking label distribution in one full pass...")

for name, ds in [("train", ds_train), ("val", ds_val), ("test", ds_test)]:
    all_labels = []
    for _, labels in ds:
        all_labels.append(labels.numpy())
    all_labels = np.concatenate(all_labels)

    unique, counts = np.unique(all_labels, return_counts=True)
    dist = {INDEX_TO_CLASS[int(u)]: int(c) for u, c in zip(unique, counts)}

    exp_dist = expected_class[name]
    dist_ok = all(dist.get(cls, 0) == exp_dist[cls] for cls in CLASS_NAMES)

    print(f"\n{name} label distribution: {dist}")
    print(f"  Expected: {exp_dist}")
    print(f"  Match: {dist_ok}")

    validation_results.append({"check": f"{name}_label_distribution", "expected": str(exp_dist), "actual": str(dist), "pass": dist_ok})


Checking label distribution in one full pass...

train label distribution: {'NV': 8824, 'MEL': 3104, 'BCC': 2243}
  Expected: {'NV': 8824, 'MEL': 3104, 'BCC': 2243}
  Match: True

val label distribution: {'NV': 1805, 'MEL': 663, 'BCC': 459}
  Expected: {'NV': 1805, 'MEL': 663, 'BCC': 459}
  Match: True

test label distribution: {'NV': 1887, 'MEL': 607, 'BCC': 463}
  Expected: {'NV': 1887, 'MEL': 607, 'BCC': 463}
  Match: True


## Validation Check 4: Visual Batch Sample


In [9]:
def visualize_batch(ds, title, save_path, n=9):
    """Display a grid of images from the first batch of a dataset."""
    for images, labels in ds.take(1):
        n = min(n, images.shape[0])
        cols = 3
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(10, 10))
        fig.suptitle(title, fontsize=16)

        for i in range(n):
            ax = axes[i // cols][i % cols]
            ax.imshow(images[i].numpy())
            cls_name = INDEX_TO_CLASS[int(labels[i].numpy())]
            ax.set_title(f"{cls_name} ({int(labels[i])})", fontsize=10)
            ax.axis("off")

        for i in range(n, rows * cols):
            axes[i // cols][i % cols].axis("off")

        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.close()
        print(f"Saved: {save_path}")
        break


visualize_batch(ds_train, "Pipeline Output: Train Batch", os.path.join(d_pipeline, "pipeline_batch_train.png"))
visualize_batch(ds_val, "Pipeline Output: Val Batch", os.path.join(d_pipeline, "pipeline_batch_val.png"))
visualize_batch(ds_test, "Pipeline Output: Test Batch", os.path.join(d_pipeline, "pipeline_batch_test.png"))


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [41.77761..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [6.816988..225.75558].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [41.443077..246.22441].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [36.618755..226.67711].
Clipping input data to the v

Saved: D:\SKIN CANCER/pipeline_output\audit_reports\data_pipeline\pipeline_batch_train.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [7.4885025..193.49777].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [7.7717896..214.80193].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [26.411606..220.62271].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [22.03003..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [14.9515705..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [13.241404..230.64282].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [12.627448..233.86661].


Saved: D:\SKIN CANCER/pipeline_output\audit_reports\data_pipeline\pipeline_batch_val.png
Saved: D:\SKIN CANCER/pipeline_output\audit_reports\data_pipeline\pipeline_batch_test.png


## Save Pipeline Config & Validation Report


In [10]:
pipeline_config = [
    {"parameter": "batch_size", "value": str(PIPELINE_BATCH_SIZE)},
    {"parameter": "shuffle_buffer", "value": str(PIPELINE_SHUFFLE_BUFFER)},
    {"parameter": "shuffle_applied_to", "value": "train only"},
    {"parameter": "augmentation_applied_to", "value": "train only"},
    {"parameter": "prefetch", "value": "AUTOTUNE"},
    {"parameter": "parallel_map_calls", "value": "AUTOTUNE"},
    {"parameter": "drop_remainder", "value": "False"},
    {"parameter": "class_weight_NV", "value": f"{CLASS_WEIGHTS[0]:.4f}"},
    {"parameter": "class_weight_MEL", "value": f"{CLASS_WEIGHTS[1]:.4f}"},
    {"parameter": "class_weight_BCC", "value": f"{CLASS_WEIGHTS[2]:.4f}"},
]

df_config = pd.DataFrame(pipeline_config)
config_path = os.path.join(d_pipeline, "pipeline_config.csv")
df_config.to_csv(config_path, index=False)
print(f"Pipeline config saved to: {config_path}")
display(df_config)


Pipeline config saved to: D:\SKIN CANCER/pipeline_output\audit_reports\data_pipeline\pipeline_config.csv


,parameter,value
0,batch_size,32
1,shuffle_buffer,4096
2,shuffle_applied_to,train only
3,augmentation_applied_to,train only
4,prefetch,AUTOTUNE
5,parallel_map_calls,AUTOTUNE
6,drop_remainder,False
7,class_weight_NV,0.5353
8,class_weight_MEL,1.5218
9,class_weight_BCC,2.1060


In [11]:
df_validation = pd.DataFrame(validation_results)
validation_path = os.path.join(d_pipeline, "pipeline_validation_report.csv")
df_validation.to_csv(validation_path, index=False)
print(f"Validation report saved to: {validation_path}")
display(df_validation)

all_passed = df_validation["pass"].all()
print(f"\nAll validation checks passed: {all_passed}")

if not all_passed:
    failed = df_validation[df_validation["pass"] == False]
    print("FAILED CHECKS:")
    display(failed)


Validation report saved to: D:\SKIN CANCER/pipeline_output\audit_reports\data_pipeline\pipeline_validation_report.csv


,check,expected,actual,pass
0,train_batch_shape,"(B,224,224,3)","(32, 224, 224, 3)",True
1,train_batch_dtype,float32,float32,True
2,train_batch_range,"[0.0, 1.0]","[0.0000, 255.0000]",False
3,train_label_range,"[0, 2]","[0, 0]",True
4,val_batch_shape,"(B,224,224,3)","(32, 224, 224, 3)",True
5,val_batch_dtype,float32,float32,True
6,val_batch_range,"[0.0, 1.0]","[0.0000, 255.0000]",False
7,val_label_range,"[0, 2]","[0, 0]",True
8,test_batch_shape,"(B,224,224,3)","(32, 224, 224, 3)",True
9,test_batch_dtype,float32,float32,True



All validation checks passed: False
FAILED CHECKS:


,check,expected,actual,pass
2,train_batch_range,"[0.0, 1.0]","[0.0000, 255.0000]",False
6,val_batch_range,"[0.0, 1.0]","[0.0000, 255.0000]",False
10,test_batch_range,"[0.0, 1.0]","[0.0000, 255.0000]",False


## Section 10 Summary


In [12]:
print("=== SECTION 10 FINAL SUMMARY ===")
print(f"Batch size: {PIPELINE_BATCH_SIZE}")
print(f"Train: {expected['train']} samples, shuffled, augmented")
print(f"Val:   {expected['val']} samples, deterministic, no augmentation")
print(f"Test:  {expected['test']} samples, deterministic, no augmentation")
print(f"")
print(f"Class weights computed (balanced):")
for idx, cls in INDEX_TO_CLASS.items():
    print(f"  {cls}: {CLASS_WEIGHTS[idx]:.4f}")
print(f"")
print(f"All validation checks passed: {all_passed}")

print("\nSaved files:")
print("- pipeline_config.csv")
print("- pipeline_validation_report.csv")
print("- pipeline_batch_train.png")
print("- pipeline_batch_val.png")
print("- pipeline_batch_test.png")

print("\n=== SECTION 10 DOWNSTREAM CONTRACT ===")
print("Section 11 must use build_dataset() to create train/val/test loaders.")
print("Section 11 must pass CLASS_WEIGHTS to model.fit(class_weight=CLASS_WEIGHTS).")
print("When a backbone is chosen, pass its preprocess_input as preprocess_fn to build_dataset().")
print("Do not rescan folders or re-split. The manifests are the authoritative source.")

print("\nSection 10 completed. No raw images were modified on disk.")


=== SECTION 10 FINAL SUMMARY ===
Batch size: 32
Train: 14171 samples, shuffled, augmented
Val:   2927 samples, deterministic, no augmentation
Test:  2957 samples, deterministic, no augmentation

Class weights computed (balanced):
  NV: 0.5353
  MEL: 1.5218
  BCC: 2.1060

All validation checks passed: False

Saved files:
- pipeline_config.csv
- pipeline_validation_report.csv
- pipeline_batch_train.png
- pipeline_batch_val.png
- pipeline_batch_test.png

=== SECTION 10 DOWNSTREAM CONTRACT ===
Section 11 must use build_dataset() to create train/val/test loaders.
Section 11 must pass CLASS_WEIGHTS to model.fit(class_weight=CLASS_WEIGHTS).
When a backbone is chosen, pass its preprocess_input as preprocess_fn to build_dataset().
Do not rescan folders or re-split. The manifests are the authoritative source.

Section 10 completed. No raw images were modified on disk.
